In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the pdf")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

ModuleNotFoundError: No module named 'langchain_community'

In [5]:
%pip install -U langchain-community pypdf


/Users/apple/Desktop/agentic-ai/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the pdf")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

Loaded 9 pages from the pdf

--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [7]:
!uv add langchain-community pypdf


Resolved 201 packages in 3.61s                                       
Prepared 5 packages in 2.71s                                             
Installed 6 packages in 65ms0.8                             
 + httpx-sse==0.4.3
 + langchain-classic==1.0.8
 + langchain-community==0.4.2
 + langchain-text-splitters==1.1.3
 + pypdf==6.21.4
 + sqlalchemy==2.1.8


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # ~150 words per chunk
    chunk_overlap=100,    # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "],  # tries paragraph → line → sentence → word
)

In [15]:
chunks  = splitter.split_documents(pages)
len(chunks)

37

In [19]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [20]:
!uv add langchain-huggingface langchain-chroma sentence-transformers

Resolved 236 packages in 3.63s                                       
Prepared 17 packages in 2m 45s                                           
Uninstalled 1 package in 38ms
Installed 17 packages in 549ms                              
 + cloudpickle==3.1.2
 - huggingface-hub==2.4.0
 + huggingface-hub==1.33.0
 + joblib==1.6.0
 + langchain-chroma==1.1.0
 + langchain-huggingface==1.2.3
 + mpmath==1.3.0
 + networkx==3.7
 + regex==2026.9.29
 + safetensors==0.8.0
 + scikit-learn==1.9.1
 + scipy==1.18.1
 + sentence-transformers==6.2.0
 + setuptools==84.2.1
 + sympy==1.14.0
 + threadpoolctl==3.7.0
 + torch==2.15.0
 + transformers==5.22.1


In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

/Users/apple/Desktop/agentic-ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1104 11:29:20.356000 5387 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 8397.25it/s]


Vector store ready. 37 vectors stored.


In [22]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)
for i, doc in enumerate(retrieved, 1):
    print(f"--- Result {i} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content)
    print()

--- Result 1 (page 6) ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network using the IMS (IP
Multimedia Subsystem) core. Benefits include HD voice quality (wideband audio at 16 kHz versus the 3.4 kHz of
legacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and

--- Result 2 (page 6) ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Options > Voice & Data and select LTE. If the option i

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# remove:  from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])
# --- LLM via Gemini API ---
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)